# Linear Regression Model Building Phase

### Load dataset and check first few rows

In [16]:
from sklearn.datasets import fetch_openml
import pandas as pd

pd.set_option("display.max_columns", None)
df = pd.read_csv("../data/cleaned/listings_features.csv")

print(df.shape)
df.head(10)

(8760, 25)


,host_is_superhost,neighbourhood_cleansed,accommodates,bathrooms_text,bedrooms,price_quote_price_per_night,minimum_nights,review_scores_rating,calculated_host_listings_count,has_license,distance_band,amenity_count,has_pool,has_hot_tub,has_gym,has_game_room,has_fire_pit,has_ev_charger,pets_allowed,has_city_skyline_view,has_free_parking,room_type_grouped,property_group,log_price,price_outlier_flag
0,t,District 25,2,1 private bath,1.0,80.00,1.0,4.94,1,0,5-10 km,27,0,0,0,0,0,0,0,0,1,Room,House,4.382027,0
1,t,District 18,2,1 bath,2.0,132.00,2.0,4.92,1,0,3-5 km,42,0,0,0,0,0,0,0,0,0,Entire place,Apartment/Condo,4.882802,0
2,t,District 19,2,1 bath,1.0,123.50,1.0,4.90,7,0,0-3 km,43,0,0,0,0,0,0,0,0,1,Entire place,Guest Unit,4.816241,0
3,t,District 6,2,1 private bath,1.0,114.00,1.0,4.86,1,0,0-3 km,29,0,0,0,0,0,0,0,0,1,Room,House,4.736198,0
4,t,District 19,2,1 bath,1.0,128.00,1.0,4.95,7,0,0-3 km,42,0,0,0,0,0,0,0,1,1,Entire place,Guest Unit,4.852030,0
5,t,District 19,2,1 bath,1.0,128.00,1.0,4.96,7,0,0-3 km,40,0,0,0,0,0,0,0,1,1,Entire place,Guest Unit,4.852030,0
6,t,District 19,4,2.5 baths,2.0,381.00,1.0,4.89,7,0,0-3 km,61,0,1,0,0,1,0,0,0,1,Entire place,House,5.942799,0
7,f,District 14,4,1 bath,1.0,NaN,2.0,4.70,1,0,10+ km,7,0,0,0,0,0,0,0,0,1,Room,House,NaN,0
8,t,District 19,2,1.5 baths,1.0,203.50,1.0,4.97,7,0,0-3 km,51,0,0,0,0,0,0,0,1,1,Entire place,Apartment/Condo,5.315666,0
9,f,District 6,2,1 bath,1.0,170.67,3.0,4.93,2,0,3-5 km,45,0,0,0,0,0,0,0,0,1,Entire place,Guest Unit,5.139732,0


### Drop missing values in target variable

In [17]:
df = df.dropna(subset=['log_price'])

# Ensure that there are no more missing target values
df['log_price'].isnull().sum()

np.int64(0)

### Convert bathroom text to numbers

Currently, the bathroom_text variable contains listings like "1.5 baths" and "shared bath" which would create an excess of dummy variables. To fix this, converting these values to a number first will reduce the amount of dummies created

In [18]:
df = df.copy()
txt = df['bathrooms_text'].str.lower()

# Number of baths: first number in the text (e.g. "1.5 baths" -> 1.5)
df['bathrooms'] = txt.str.extract(r'(\d+\.?\d*)')[0].astype(float)

# "Half-bath" / "Shared half-bath" have no number, so count them as 0.5
df.loc[txt.str.contains('half', na=False) & df['bathrooms'].isnull(), 'bathrooms'] = 0.5

# 1 if the bath is shared, 0 otherwise
df['shared_bath'] = txt.str.contains('shared', na=False).astype(int)

# The original text column is now replaced
df = df.drop(columns=['bathrooms_text'])

# Check the result: any remaining blanks (imputed later, after the split) and the value counts
print("Missing bathrooms:", df['bathrooms'].isnull().sum())
print(df['bathrooms'].value_counts().sort_index())
print(df['shared_bath'].value_counts())

Missing bathrooms: 12
bathrooms
0.0        9
0.5        3
1.0     3187
1.5      179
2.0     1436
2.5      540
3.0      607
3.5      695
4.0      740
4.5      242
5.0       80
5.5       36
6.0       79
6.5       32
7.0       67
7.5       15
8.0      106
8.5        9
9.0       41
9.5        8
10.0      13
10.5       5
11.0       1
12.0      15
12.5       3
13.0       5
13.5       2
14.0       4
15.0       3
16.0       6
16.5       2
17.0       1
17.5       1
18.0       2
19.0       1
20.0       2
21.0       1
23.0       1
24.0       2
40.5       1
47.0       1
50.0       2
Name: count, dtype: int64
shared_bath
0    8129
1      68
Name: count, dtype: int64


### Train/Test Split

In [19]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size = 0.25, random_state = 1234)

### Set target variable and drop-non predictors

In [20]:
import statsmodels.api as sm

predictors = train.drop(columns=['log_price','price_quote_price_per_night','price_outlier_flag'])

X = predictors.copy()
y = train['log_price']

### Impute missing values (Continous Variables)

In [21]:
num_cols = X.select_dtypes(include='number').columns

# Store the train medians so the same values can fill missing data in the test set later
medians = {}

for col in num_cols:
    if X[col].isnull().any():
        # Create missing flag column
        X[f'{col}_was_missing'] = X[col].isnull().astype(int)
        
        # Impute with median
        median = X[col].median()
        medians[col] = median
        X[col] = X[col].fillna(median)

print(medians)

{'bedrooms': np.float64(2.0), 'minimum_nights': np.float64(1.0), 'review_scores_rating': np.float64(4.91), 'bathrooms': np.float64(2.0)}


### Create dummy variables and ensure no missing values remain

In [22]:
X = pd.get_dummies(X, drop_first=True)
X = X.astype(float)

print(X.shape)
print("Missing values left:", X.isnull().sum().sum())

(6147, 66)
Missing values left: 0


### Univariate Statistical Testing

In [23]:
from sklearn.feature_selection import SelectKBest, f_regression

# Fit SelectKBest
selector = SelectKBest(score_func=f_regression, k='all') 
selector.fit(X, y)

# Create a DataFrame with feature names, F-scores, and p-values
scores_df = pd.DataFrame({
    'Feature': X.columns,
    'F_score': selector.scores_,
    'p_value': selector.pvalues_
})

# Sort by F-score descending
scores_df = scores_df.sort_values(by='F_score', ascending=False)

with pd.option_context('display.max_rows', None):
    print(scores_df)

                               Feature      F_score       p_value
0                         accommodates  5304.615485  0.000000e+00
16                           bathrooms  4941.029877  0.000000e+00
1                             bedrooms  4893.120545  0.000000e+00
63                property_group_House   313.451927  1.893038e-68
14               has_city_skyline_view   269.561974  2.553842e-59
61           property_group_Guest Unit   262.883490  6.344108e-58
6                        amenity_count   229.650909  5.870485e-51
28  neighbourhood_cleansed_District 15   224.589684  6.810162e-50
60              room_type_grouped_Room   196.663117  5.296479e-44
59               distance_band_5-10 km   185.954278  9.791307e-42
65            property_group_Townhouse   159.637688  3.810508e-36
57                distance_band_10+ km   142.561156  1.674090e-32
8                          has_hot_tub   124.032579  1.552177e-28
17                         shared_bath    97.743406  7.051702e-23
33   neigh

In [24]:
selector = SelectKBest(score_func=f_regression, k='all')  # 'all' keeps all features
selector.fit(X, y)

# Create a DataFrame of scores and p-values
pval_df = pd.DataFrame({
    'Feature': X.columns,
    'F_score': selector.scores_,
    'p_value': selector.pvalues_
})

# Filter for features with p-value < 0.001
# p-value determined relative to sample size (Raftery)
selected_features = pval_df[pval_df['p_value'] < 0.001]['Feature']

# Create a new DataFrame with only those selected columns
X_reduced = X[selected_features.tolist()]

X_reduced.head()

,accommodates,bedrooms,minimum_nights,review_scores_rating,calculated_host_listings_count,has_license,amenity_count,has_pool,has_hot_tub,has_gym,has_game_room,has_fire_pit,pets_allowed,has_city_skyline_view,bathrooms,shared_bath,review_scores_rating_was_missing,bathrooms_was_missing,host_is_superhost_t,neighbourhood_cleansed_District 10,neighbourhood_cleansed_District 13,neighbourhood_cleansed_District 15,neighbourhood_cleansed_District 16,neighbourhood_cleansed_District 17,neighbourhood_cleansed_District 19,neighbourhood_cleansed_District 2,neighbourhood_cleansed_District 26,neighbourhood_cleansed_District 29,neighbourhood_cleansed_District 32,neighbourhood_cleansed_District 33,neighbourhood_cleansed_District 34,neighbourhood_cleansed_District 5,neighbourhood_cleansed_District 7,neighbourhood_cleansed_District 9,distance_band_10+ km,distance_band_3-5 km,distance_band_5-10 km,room_type_grouped_Room,property_group_Guest Unit,property_group_Hotel/Resort,property_group_House,property_group_Other,property_group_Townhouse
7461,12.0,4.0,1.0,4.94,73.0,0.0,61.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
523,6.0,2.0,1.0,4.76,56.0,0.0,47.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7484,4.0,1.0,1.0,4.94,254.0,1.0,45.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4581,10.0,3.0,2.0,4.44,1.0,0.0,36.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3745,10.0,3.0,1.0,4.89,48.0,1.0,63.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,2.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [25]:
# Fit linear regression model
X_reduced = sm.add_constant(X_reduced)
model = sm.OLS(y, X_reduced)
result = model.fit()

# Print summary
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.628
Model:                            OLS   Adj. R-squared:                  0.626
Method:                 Least Squares   F-statistic:                     239.9
Date:                Mon, 09 Nov 2026   Prob (F-statistic):               0.00
Time:                        18:55:06   Log-Likelihood:                -2746.4
No. Observations:                6147   AIC:                             5581.
Df Residuals:                    6103   BIC:                             5877.
Df Model:                          43                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

### Variable Reduction

#### Backward Selection

In [28]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LinearRegression
import pandas as pd

# Model
lr = LinearRegression()

# Backward selection to find best number of features
sfs_backward = SFS(lr,
          k_features="best",
          forward=False,
          floating=False,
          scoring='neg_mean_squared_error',
          cv=10)

sfs_backward = sfs_backward.fit(X_reduced, y)

# Get selected feature names
selected_features = list(sfs_backward.k_feature_names_)
print("Selected features:", selected_features)

Selected features: ['const', 'accommodates', 'bedrooms', 'minimum_nights', 'review_scores_rating', 'calculated_host_listings_count', 'has_license', 'amenity_count', 'has_pool', 'has_hot_tub', 'has_gym', 'has_game_room', 'pets_allowed', 'has_city_skyline_view', 'bathrooms', 'shared_bath', 'review_scores_rating_was_missing', 'host_is_superhost_t', 'neighbourhood_cleansed_District 13', 'neighbourhood_cleansed_District 15', 'neighbourhood_cleansed_District 16', 'neighbourhood_cleansed_District 17', 'neighbourhood_cleansed_District 19', 'neighbourhood_cleansed_District 2', 'neighbourhood_cleansed_District 26', 'neighbourhood_cleansed_District 29', 'neighbourhood_cleansed_District 33', 'neighbourhood_cleansed_District 34', 'neighbourhood_cleansed_District 5', 'neighbourhood_cleansed_District 9', 'distance_band_10+ km', 'distance_band_3-5 km', 'distance_band_5-10 km', 'room_type_grouped_Room', 'property_group_Guest Unit', 'property_group_Hotel/Resort', 'property_group_House', 'property_group_

In [29]:
selected_features = list(sfs_backward.k_feature_names_)

# Subset the DataFrame to selected features
X_selected = X_reduced[selected_features].copy()
X_selected = sm.add_constant(X_selected)
model = sm.OLS(y, X_selected).fit()

# Summary with p-values, coefficients, R², etc.
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     270.3
Date:                Mon, 09 Nov 2026   Prob (F-statistic):               0.00
Time:                        19:02:44   Log-Likelihood:                -2756.9
No. Observations:                6147   AIC:                             5592.
Df Residuals:                    6108   BIC:                             5854.
Df Model:                          38                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

#### Forward Selection

In [30]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LinearRegression
import pandas as pd

# Model
lr = LinearRegression()

# Forward selection to find best number of features
sfs_forward = SFS(lr,
          k_features="best",
          forward=True,
          floating=False,
          scoring='neg_mean_squared_error',
          cv=10)

sfs_forward = sfs_forward.fit(X_reduced, y)

# Get selected feature names
selected_features = list(sfs_forward.k_feature_names_)
print("Selected features:", selected_features)

Selected features: ['const', 'accommodates', 'bedrooms', 'minimum_nights', 'review_scores_rating', 'calculated_host_listings_count', 'has_license', 'amenity_count', 'has_pool', 'has_hot_tub', 'has_gym', 'has_game_room', 'pets_allowed', 'has_city_skyline_view', 'bathrooms', 'shared_bath', 'review_scores_rating_was_missing', 'host_is_superhost_t', 'neighbourhood_cleansed_District 13', 'neighbourhood_cleansed_District 15', 'neighbourhood_cleansed_District 16', 'neighbourhood_cleansed_District 17', 'neighbourhood_cleansed_District 19', 'neighbourhood_cleansed_District 2', 'neighbourhood_cleansed_District 26', 'neighbourhood_cleansed_District 29', 'neighbourhood_cleansed_District 33', 'neighbourhood_cleansed_District 34', 'neighbourhood_cleansed_District 5', 'neighbourhood_cleansed_District 9', 'distance_band_10+ km', 'distance_band_3-5 km', 'distance_band_5-10 km', 'room_type_grouped_Room', 'property_group_Guest Unit', 'property_group_Hotel/Resort', 'property_group_House', 'property_group_

In [31]:
selected_features = list(sfs_forward.k_feature_names_)

# Subset the DataFrame to selected features
X_selected = X_reduced[selected_features].copy()
X_selected = sm.add_constant(X_selected)
model = sm.OLS(y, X_selected).fit()

# Summary with p-values, coefficients, R², etc.
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     270.3
Date:                Mon, 09 Nov 2026   Prob (F-statistic):               0.00
Time:                        19:05:33   Log-Likelihood:                -2756.9
No. Observations:                6147   AIC:                             5592.
Df Residuals:                    6108   BIC:                             5854.
Df Model:                          38                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

#### Stepwise

In [32]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.linear_model import LinearRegression
import pandas as pd

# Model
lr = LinearRegression()

# Stepwise selection to find best number of features
sfs_step = SFS(lr,
          k_features="best",
          forward=True,
          floating=True,
          scoring='neg_mean_squared_error',
          cv=10)

sfs_step = sfs_step.fit(X_reduced, y)

# Get selected feature names
selected_features = list(sfs_step.k_feature_names_)
print("Selected features:", selected_features)

Selected features: ['const', 'accommodates', 'bedrooms', 'minimum_nights', 'review_scores_rating', 'calculated_host_listings_count', 'has_license', 'amenity_count', 'has_pool', 'has_hot_tub', 'has_gym', 'has_game_room', 'pets_allowed', 'has_city_skyline_view', 'bathrooms', 'shared_bath', 'review_scores_rating_was_missing', 'host_is_superhost_t', 'neighbourhood_cleansed_District 13', 'neighbourhood_cleansed_District 15', 'neighbourhood_cleansed_District 16', 'neighbourhood_cleansed_District 17', 'neighbourhood_cleansed_District 19', 'neighbourhood_cleansed_District 2', 'neighbourhood_cleansed_District 26', 'neighbourhood_cleansed_District 29', 'neighbourhood_cleansed_District 33', 'neighbourhood_cleansed_District 34', 'neighbourhood_cleansed_District 5', 'neighbourhood_cleansed_District 9', 'distance_band_10+ km', 'distance_band_3-5 km', 'distance_band_5-10 km', 'room_type_grouped_Room', 'property_group_Guest Unit', 'property_group_Hotel/Resort', 'property_group_House', 'property_group_

In [33]:
selected_features = list(sfs_step.k_feature_names_)

# Subset the DataFrame to selected features
X_selected = X_reduced[selected_features].copy()
X_selected = sm.add_constant(X_selected)
model = sm.OLS(y, X_selected).fit()

# Summary with p-values, coefficients, R², etc.
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.627
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     270.3
Date:                Mon, 09 Nov 2026   Prob (F-statistic):               0.00
Time:                        19:09:03   Log-Likelihood:                -2756.9
No. Observations:                6147   AIC:                             5592.
Df Residuals:                    6108   BIC:                             5854.
Df Model:                          38                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons